# Exploring the dataset

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
sns.set_color_codes("pastel")

In [ ]:
ds = pd.read_csv("dataset.csv")

In [ ]:
ds.columns

In [ ]:
# The label
sns.histplot(data=ds, x="age", palette="dark:b", hue="hospital_expire_flag", kde=True);

In [ ]:
sns.relplot(
    data=ds,
    x="lactate_mean",
    y="creatinine_mean",
    hue="hospital_expire_flag",
    alpha=0.4,
);

In [ ]:
corr_features = ds[
    [
        "gender",
        "glucose_min",
        "glucose_max",
        "glucose_mean",
        "lactate_min",
        "lactate_max",
        "lactate_mean",
        "creatinine_min",
        "creatinine_max",
        "creatinine_mean",
        "heart_rate_mean",
        "heart_rate_min",
        "heart_rate_max",
        "blood_pressure_mean",
        "blood_pressure_min",
        "blood_pressure_max",
        "resp_rate_mean",
        "resp_rate_min",
        "resp_rate_max",
        "temp_f_mean",
        "temp_f_min",
        "temp_f_max",
        "spO2_mean",
        "spO2_min",
        "spO2_max",
    ]
]
corr_features["gender"] = corr_features["gender"].map({"M": 0, "F": 1})

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(data=corr_features.corr())

In [ ]:
train_ds = pd.read_csv("ds_train.csv")
val_ds = pd.read_csv("ds_val.csv")
test_ds = pd.read_csv("ds_test.csv")

In [ ]:
train_ds["purpose"] = "training"
val_ds["purpose"] = "validation"
test_ds["purpose"] = "test"
combined_ds = pd.concat([train_ds, val_ds, test_ds])
combined_ds.columns

In [ ]:
def plot_continuous_feature(feat: str, ax=None):
    sns.kdeplot(
        data=combined_ds,
        x=feat,
        hue="purpose",
        common_norm=False,
        fill=False,
        alpha=0.5,
        palette={"training": "blue", "validation": "orange", "test": "green"},
        ax=ax,
    )

In [ ]:
plot_continuous_feature("age")

In [ ]:
wanted_features = [
    "glucose_mean",
    "lactate_mean",
    "creatinine_mean",
    "heart_rate_mean",
    "age",
]
fig, axes = plt.subplots(
    nrows=len(wanted_features) // 2,
    ncols=2 if len(wanted_features) % 2 == 0 else 3,
    figsize=(15, 10),
)
axes = axes.flatten()

for i, feat in enumerate(wanted_features):
    plot_continuous_feature(feat, axes[i])
    axes[i].set_title(f"Distribution of {feat}")

plt.tight_layout()
plt.show()

In [ ]:
print(
    "Checking if the reason why lactate is missing is that there was no need to measure it (the patient wasn't that bad)"
)
mortality_by_lactate = (
    combined_ds.groupby("lactate_missing")["hospital_expire_flag"].mean() * 100
)
print(
    f"Mortality in patients where lactate WAS measured: {mortality_by_lactate[0]:.2f}%"
)
print(
    f"Mortality in patients where lactate WAS NOT measured: {mortality_by_lactate[1]:.2f}%"
)
print(f"Difference: {abs(mortality_by_lactate[0] - mortality_by_lactate[1]):.2f}%")

plt.figure(figsize=(8, 6))
ax = sns.barplot(
    data=combined_ds,
    x="lactate_missing",
    y="hospital_expire_flag",
    palette=["red", "blue"],
    capsize=0.1,
)
plt.title("Mortality relationship with lactate measurement missing")
plt.xlabel("Lactate Missing")
plt.ylabel("Mortality")

ax.set_xticks([0, 1])
ax.set_xticklabels(["Measured (high risk)", "Not Measured (lower risk)"])
plt.tight_layout()
plt.show()

In [ ]:
combined_no_purpose = combined_ds.drop("purpose", axis=1)
mortality_correl = combined_no_purpose.corr()["hospital_expire_flag"].sort_values(
    ascending=False
)
top_mortality_correl = mortality_correl.head(10)
top_mortality_correl[1:]

In [ ]:
plt.figure(figsize=(12, 10))
sns.heatmap(combined_no_purpose.corr())